# Notebook 6 - Large LLM Comparison on Google Colab

## Goal

This notebook is designed for Google Colab Pro or Pro+. It compares the same truth-probing experiment across multiple LLMs.

The professor asked for different results on different LLMs. This notebook produces the table needed for that comparison.

## What this notebook compares

Default enabled models:

- `microsoft/phi-2` as a small baseline,
- `mistralai/Mistral-7B-Instruct-v0.3`,
- `meta-llama/Meta-Llama-3-8B-Instruct` if your Hugging Face account has accepted access.

Optional models:

- `meta-llama/Llama-2-7b-chat-hf`, useful if you have Llama 2 access but not Llama 3 access,
- `meta-llama/Llama-2-13b-chat-hf`, close to the model family used in the original `repeng` comparison,
- `meta-llama/Llama-3.3-70B-Instruct`,
- `meta-llama/Meta-Llama-3-70B-Instruct`.

The notebook saves CSV and PNG results both inside the cloned repository and, when Google Drive is mounted, inside `MyDrive/Lie-Detector-for-LLM-results/`.


## Step 1 - Runtime setup

In Colab, select a GPU runtime before running this notebook:

`Runtime -> Change runtime type -> Hardware accelerator -> GPU`

Recommended modes:

- `RUN_MODE = "standard"`: Phi-2, Mistral 7B, Llama 3 8B if access is granted, and Llama 2 13B on large enough GPUs.
- `RUN_MODE = "llama2"`: Llama 2 7B and 13B only, useful if your token has Llama 2 access but not Llama 3 access.
- `RUN_MODE = "llama70b"`: Llama 70B models only. Use an A100 80GB or larger runtime. A100 40GB is usually not enough for 70B in this hidden-state extraction setup.

Run the cells from top to bottom. The first code cell clones the whole GitHub repository into `/content/Lie-Detector-for-LLM`, so Colab is not only running the single notebook file.


In [ ]:
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path

REPO_URL = "https://github.com/imadsharof/Lie-Detector-for-LLM.git"
REPO_DIR = Path("/content/Lie-Detector-for-LLM")
MOUNT_GOOGLE_DRIVE = True
RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
DRIVE_RESULTS_DIR = None

# Change this before running all cells.
# Options: "standard", "llama2", "llama70b".
RUN_MODE = "standard"

# Keep this False for normal runs. Set True only if you knowingly want to try
# 70B on a smaller GPU with a high risk of out-of-memory failure.
ALLOW_LOW_MEMORY_70B_ATTEMPT = False

if "google.colab" in sys.modules:
    if MOUNT_GOOGLE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_RESULTS_DIR = Path("/content/drive/MyDrive/Lie-Detector-for-LLM-results") / RUN_ID
        DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Drive results directory: {DRIVE_RESULTS_DIR}")

    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

    # Colab images sometimes contain older packages. Phi-2 needs transformers >= 4.37,
    # and Mistral v0.3 is safest with a recent transformers release.
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "transformers>=4.42,<5",
            "accelerate>=0.33",
            "bitsandbytes>=0.43.1",
            "huggingface-hub",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[colab]"], check=True)
else:
    print("Not running inside Colab. Using the current local repository.")

print(f"RUN_ID   : {RUN_ID}")
print(f"RUN_MODE : {RUN_MODE}")


## Step 2 - Import the package and inspect the GPU

The experiment needs access to hidden states, so it loads models through Hugging Face Transformers rather than using an API endpoint.


In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "lie_detector_llm").exists():
            return candidate
    raise RuntimeError("Could not find the project root. Run this notebook from the repository.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU for large-model comparison.")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_TOTAL_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"GPU name        : {GPU_NAME}")
print(f"GPU memory (GB) : {GPU_TOTAL_GB:.1f}")


## Step 3 - Hugging Face login

Llama models are gated. A token that works for Llama 2 does **not** automatically work for Llama 3. You must accept access on the exact model page you want to load.

Before running this step:

1. create a Hugging Face account,
2. open the model page, for example `meta-llama/Meta-Llama-3-8B-Instruct`,
3. click the access/license agreement and wait until access is granted,
4. create a token with read access,
5. store it in Colab secrets as `HF_TOKEN`, with Notebook access enabled.

Do not paste the token into the notebook output. Keep it in Colab Secrets.


In [ ]:
import os
from huggingface_hub import HfApi, login, notebook_login

hf_token = os.environ.get("HF_TOKEN")

try:
    from google.colab import userdata
    hf_token = hf_token or userdata.get("HF_TOKEN")
except Exception:
    pass

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token)
    user = HfApi(token=hf_token).whoami().get("name")
    print(f"Logged in to Hugging Face as: {user}")
else:
    print("No HF_TOKEN found. A login widget will appear.")
    notebook_login()


## Step 4 - Build the shared evaluation datasets

Every model is evaluated on the same prompts, datasets, train split, test split, probe method, and layer rule. This makes the comparison fair.

`MAX_HF_GROUPS` controls the number of Hugging Face benchmark groups. Increase it for a stronger final run, but start small to verify the runtime.

For 70B runs, the notebook uses fewer groups by default because loading and evaluating the model is much slower and requires much more memory.


In [ ]:
from lie_detector_llm.datasets import build_dataset_collection

INCLUDE_HF_DATASETS = True
MAX_HF_GROUPS = 10 if RUN_MODE == "llama70b" else 25

collection = build_dataset_collection(
    include_hf_datasets=INCLUDE_HF_DATASETS,
    max_hf_groups=MAX_HF_GROUPS,
)
dataset_names = collection.dataset_names()

stats = (
    collection.frame.groupby("dataset_name")
    .agg(rows=("dataset_name", "size"), groups=("group_id", "nunique"))
    .reset_index()
    .sort_values("dataset_name")
)

display(stats)
print(f"RUN_MODE       : {RUN_MODE}")
print(f"MAX_HF_GROUPS  : {MAX_HF_GROUPS}")
print(f"Total prompts  : {len(collection.frame)}")
print(f"Total groups   : {collection.frame['group_id'].nunique()}")


## Step 5 - Choose Models To Compare

The selected models depend on `RUN_MODE` from Step 1.

- `standard`: runs a practical comparison for the report.
- `llama2`: runs Llama 2 models only.
- `llama70b`: runs Llama 70B models only. Use A100 80GB or larger. On A100 40GB, leave `ALLOW_LOW_MEMORY_70B_ATTEMPT=False` so the notebook fails early with a clear message instead of wasting time.

All 7B/8B/13B/70B models use 4-bit loading to reduce memory usage.


In [ ]:
from huggingface_hub import HfApi

CHECK_HF_ACCESS_BEFORE_RUNNING = True
ACCESS_FAILURES = []

ALL_MODEL_CONFIGS = [
    {
        "label": "Phi-2 baseline",
        "model_name": "microsoft/phi-2",
        "load_in_4bit": False,
        "modes": ["standard"],
    },
    {
        "label": "Mistral 7B Instruct",
        "model_name": "mistralai/Mistral-7B-Instruct-v0.3",
        "load_in_4bit": True,
        "modes": ["standard"],
    },
    {
        "label": "Llama 3 8B Instruct",
        "model_name": "meta-llama/Meta-Llama-3-8B-Instruct",
        "load_in_4bit": True,
        "modes": ["standard"],
    },
    {
        "label": "Llama 2 7B Chat",
        "model_name": "meta-llama/Llama-2-7b-chat-hf",
        "load_in_4bit": True,
        "modes": ["llama2"],
    },
    {
        "label": "Llama 2 13B Chat",
        "model_name": "meta-llama/Llama-2-13b-chat-hf",
        "load_in_4bit": True,
        "modes": ["standard", "llama2"],
        "min_gpu_gb": 24,
    },
    {
        "label": "Llama 3.3 70B Instruct",
        "model_name": "meta-llama/Llama-3.3-70B-Instruct",
        "load_in_4bit": True,
        "modes": ["llama70b"],
        "min_gpu_gb": 70,
    },
    {
        "label": "Llama 3 70B Instruct",
        "model_name": "meta-llama/Meta-Llama-3-70B-Instruct",
        "load_in_4bit": True,
        "modes": ["llama70b"],
        "min_gpu_gb": 70,
    },
]

if RUN_MODE not in {"standard", "llama2", "llama70b"}:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")

if RUN_MODE == "llama70b" and GPU_TOTAL_GB < 70 and not ALLOW_LOW_MEMORY_70B_ATTEMPT:
    raise RuntimeError(
        f"RUN_MODE='llama70b' requested, but the current GPU has only {GPU_TOTAL_GB:.1f} GB. "
        "Use an A100 80GB/H100 runtime, or set ALLOW_LOW_MEMORY_70B_ATTEMPT=True if you want to try anyway."
    )

active_configs = []
for config in ALL_MODEL_CONFIGS:
    if RUN_MODE not in config["modes"]:
        continue
    min_gpu_gb = config.get("min_gpu_gb")
    if min_gpu_gb and GPU_TOTAL_GB < min_gpu_gb and not (RUN_MODE == "llama70b" and ALLOW_LOW_MEMORY_70B_ATTEMPT):
        print(f"Skipping {config['label']}: needs about {min_gpu_gb} GB GPU memory, current GPU has {GPU_TOTAL_GB:.1f} GB.")
        continue
    active_configs.append(config)

if CHECK_HF_ACCESS_BEFORE_RUNNING:
    api = HfApi(token=os.environ.get("HF_TOKEN"))
    checked_configs = []
    for config in active_configs:
        try:
            api.model_info(config["model_name"], token=os.environ.get("HF_TOKEN"))
            checked_configs.append(config)
        except Exception as exc:
            ACCESS_FAILURES.append(
                {
                    "model_label": config["label"],
                    "model_name": config["model_name"],
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                    "run_mode": RUN_MODE,
                    "gpu_name": GPU_NAME,
                    "gpu_total_gb": GPU_TOTAL_GB,
                }
            )
            print(f"Skipping {config['label']} because Hugging Face access failed: {exc}")
    active_configs = checked_configs

print("Models that will run:")
for config in active_configs:
    print(f"- {config['label']}: {config['model_name']} (4-bit={config['load_in_4bit']})")

if not active_configs:
    raise RuntimeError("No model is enabled and accessible. Check RUN_MODE, HF_TOKEN, accepted licenses, and GPU memory.")


## Step 6 - Configure the shared probing experiment

We use one fixed setup for all models:

- train dataset: `repeng_truthful`,
- evaluation: test split of every dataset,
- probe method: logistic regression,
- layer index: `-1`, the final transformer layer,
- activation batch size: 1, safer for large models.

A stronger final report can add a separate layer sweep per model, but this fixed-layer comparison is the cleanest first multi-LLM result.


In [ ]:
TRAIN_DATASET = "repeng_truthful"
PROBE_METHOD = "lr"
LAYER_INDEX = -1
ACTIVATION_BATCH_SIZE = 1
SPLIT_EVALUATION = True

print("Shared experiment settings")
print(f"Run mode      : {RUN_MODE}")
print(f"Train dataset : {TRAIN_DATASET}")
print(f"Eval datasets : {dataset_names}")
print(f"Probe method  : {PROBE_METHOD}")
print(f"Layer index   : {LAYER_INDEX}")


## Step 7 - Run the comparison loop

Each model is loaded, evaluated, and then removed from the model cache before the next model starts. This is important on Colab because GPU memory is limited.


In [ ]:
import gc
import time
import pandas as pd
import torch

from lie_detector_llm.experiment import run_transfer_experiment
from lie_detector_llm.models import clear_model_cache

all_results = []
failures = list(ACCESS_FAILURES)

for config in active_configs:
    print("=" * 80)
    print(f"Running {config['label']} ({config['model_name']})")
    start = time.perf_counter()
    try:
        output = run_transfer_experiment(
            collection=collection,
            train_dataset_name=TRAIN_DATASET,
            eval_dataset_names=dataset_names,
            model_name=config["model_name"],
            probe_method=PROBE_METHOD,
            layer_index=LAYER_INDEX,
            split_evaluation=SPLIT_EVALUATION,
            activation_batch_size=ACTIVATION_BATCH_SIZE,
            load_in_4bit=config["load_in_4bit"],
            show_progress=True,
        )
        df_model = output.summary_table()
        df_model["model_label"] = config["label"]
        df_model["load_in_4bit"] = config["load_in_4bit"]
        df_model["runtime_seconds"] = time.perf_counter() - start
        df_model["run_id"] = RUN_ID
        df_model["run_mode"] = RUN_MODE
        df_model["gpu_name"] = GPU_NAME
        df_model["gpu_total_gb"] = GPU_TOTAL_GB
        df_model["max_hf_groups"] = MAX_HF_GROUPS
        all_results.append(df_model)
        display(df_model.sort_values("eval_dataset"))
    except Exception as exc:
        failures.append(
            {
                "model_label": config["label"],
                "model_name": config["model_name"],
                "error_type": type(exc).__name__,
                "error": str(exc),
                "run_id": RUN_ID,
                "run_mode": RUN_MODE,
                "gpu_name": GPU_NAME,
                "gpu_total_gb": GPU_TOTAL_GB,
                "max_hf_groups": MAX_HF_GROUPS,
            }
        )
        print(f"FAILED: {type(exc).__name__}: {exc}")
    finally:
        clear_model_cache()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if not all_results:
    raise RuntimeError("No model completed successfully.")

comparison = pd.concat(all_results, ignore_index=True)
failures_df = pd.DataFrame(failures)


## Step 8 - Build the comparison table


In [ ]:
pivot = comparison.pivot_table(
    index="model_label",
    columns="eval_dataset",
    values="grouped_accuracy",
).round(3)

display(pivot)

transfer_only = comparison[comparison["eval_dataset"] != TRAIN_DATASET]
summary = (
    transfer_only.groupby(["model_label", "model_name", "run_mode", "gpu_name"], as_index=False)
    .agg(
        mean_transfer_accuracy=("grouped_accuracy", "mean"),
        min_transfer_accuracy=("grouped_accuracy", "min"),
        max_transfer_accuracy=("grouped_accuracy", "max"),
        runtime_seconds=("runtime_seconds", "max"),
        max_hf_groups=("max_hf_groups", "max"),
    )
    .sort_values("mean_transfer_accuracy", ascending=False)
)

display(summary)

if len(failures_df):
    print("Models that failed:")
    display(failures_df)


## Step 9 - Plot model comparison


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(
    data=transfer_only,
    x="eval_dataset",
    y="grouped_accuracy",
    hue="model_label",
    ax=ax,
)
ax.set_ylim(0, 1)
ax.set_title("Cross-dataset transfer accuracy by model")
ax.set_xlabel("Evaluation dataset")
ax.set_ylabel("Grouped accuracy")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig


## Step 10 - Save the results

This cell writes the CSV files and the plot inside the Colab clone of the repository. If Google Drive was mounted in Step 1, it also writes the same files to `MyDrive/Lie-Detector-for-LLM-results/<run_id>/`, which persists after the Colab runtime shuts down.


In [ ]:
from pathlib import Path
import shutil

results_dir = PROJECT_ROOT / "results" / "large_llm_comparison" / RUN_ID
results_dir.mkdir(parents=True, exist_ok=True)

comparison_path = results_dir / "large_llm_comparison.csv"
summary_path = results_dir / "large_llm_comparison_summary.csv"
failures_path = results_dir / "large_llm_comparison_failures.csv"
plot_path = results_dir / "large_llm_comparison_transfer.png"

comparison.to_csv(comparison_path, index=False)
summary.to_csv(summary_path, index=False)
failures_df.to_csv(failures_path, index=False)
fig.savefig(plot_path, dpi=160, bbox_inches="tight")

print(f"Saved detailed comparison to: {comparison_path}")
print(f"Saved summary to            : {summary_path}")
print(f"Saved failures to           : {failures_path}")
print(f"Saved plot to               : {plot_path}")

if DRIVE_RESULTS_DIR is not None:
    DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    for path in [comparison_path, summary_path, failures_path, plot_path]:
        shutil.copy2(path, DRIVE_RESULTS_DIR / path.name)
    print(f"Copied persistent results to: {DRIVE_RESULTS_DIR}")


## Step 11 - Optional full matrix for the best model

The comparison above trains only on `repeng_truthful`. After identifying the best model, you can run a full train-dataset by eval-dataset matrix for that model.

This is slower, so it is disabled by default.


In [ ]:
RUN_FULL_MATRIX_FOR_BEST = False

if RUN_FULL_MATRIX_FOR_BEST:
    from lie_detector_llm.experiment import run_full_transfer_matrix
    from lie_detector_llm.plotting import plot_transfer_heatmap

    best_row = summary.iloc[0]
    best_config = next(
        config for config in active_configs if config["label"] == best_row["model_label"]
    )

    matrix = run_full_transfer_matrix(
        collection=collection,
        model_name=best_config["model_name"],
        probe_method=PROBE_METHOD,
        layer_index=LAYER_INDEX,
        split_evaluation=True,
        activation_batch_size=ACTIVATION_BATCH_SIZE,
        load_in_4bit=best_config["load_in_4bit"],
        show_progress=True,
    )

    display(matrix.summary_table())
    fig, ax = plot_transfer_heatmap(
        matrix.results,
        title=f"Full transfer matrix: {best_config['label']}",
    )
    display(fig)

    clear_model_cache()


## How to report these results

A good course-report paragraph should state:

- which models were compared,
- which datasets were used,
- which probe and layer were fixed for fairness,
- the average off-domain grouped accuracy for each model,
- whether larger instruction-tuned models improved transfer.

The key table is `large_llm_comparison_summary.csv`. The key plot is the bar chart from Step 9.
